# Colocamos las liberías a utilizar

In [21]:
import dash
from dash import dcc, html, Input, Output
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Preprocesamiento de datos

In [22]:
print("Cargando y procesando datos...")

# Diccionario maestro ROBUSTO para evitar pérdidas de datos por cambios de nombre del DEIS
mapa_sss = {
    'Osorno': ['Osorno'],
    'Chiguayante': ['Concepción', 'Concepcion'],
    'Concepcion': ['Concepción', 'Concepcion'],
    'Temuco': ['Araucanía Sur', 'Araucania Sur'],
    'Valdivia': ['Valdivia', 'De Los Ríos', 'Los Ríos', 'Los Rios'] # Captura todas las variaciones
}

# --- 2.1 Datos de Urgencias Respiratorias ---
lista_urg = []
for year in range(2020, 2026):
    try:
        df = pd.read_csv(f'Atenciones_Respiratorias/AtencionesRespiratorias{year}.csv')
        if 'ComunaGlosa' in df.columns:
            df['Comuna'] = df['ComunaGlosa']
        elif 'NombreComuna' in df.columns:
            df['Comuna'] = df['NombreComuna']
        else:
            df['Comuna'] = 'Desconocida'
            
        df['Comuna'] = df['Comuna'].astype(str).str.strip().str.replace('Concepción', 'Concepcion')
        df['Año'] = year
        lista_urg.append(df)
    except Exception as e:
        print(f"No se pudo procesar el año {year}: {e}")

df_urgencias = pd.concat(lista_urg, ignore_index=True) if lista_urg else pd.DataFrame()
if not df_urgencias.empty:
    df_urgencias['fecha'] = pd.to_datetime(df_urgencias['fecha'], format='%d/%m/%Y', errors='coerce')
    df_urgencias['Mes'] = df_urgencias['fecha'].dt.month
    df_urgencias['semana'] = df_urgencias['fecha'].dt.isocalendar().week

# --- 2.2 Datos de Contaminación PM 2.5 ---
ciudades_pm = ['Osorno', 'Chiguayante', 'Concepcion', 'Temuco', 'Valdivia']
lista_pm = []
for ciudad in ciudades_pm:
    try:
        df_c = pd.read_csv(f'Datos_PM/{ciudad}.csv', sep=';', encoding='latin-1') 
        df_c['Comuna'] = ciudad
        df_c['fecha'] = pd.to_datetime(df_c['FECHA (YYMMDD)'], format='%y%m%d', errors='coerce')
        df_c.rename(columns={'Registros validados': 'PM25'}, inplace=True)
        df_c['PM25'] = pd.to_numeric(df_c['PM25'].astype(str).str.replace(',', '.'), errors='coerce')
        lista_pm.append(df_c)
    except:
        pass
df_pm25 = pd.concat(lista_pm, ignore_index=True) if lista_pm else pd.DataFrame()

# --- 2.3 Datos de Camas Críticas (REM20) ---
try:
    df_camas = pd.read_csv('Indicadores de Salud REM20/Indicadores_REM20.csv')
    df_camas = df_camas[df_camas['AREA_FUNCIONAL'].str.contains('Intensiv|Crític', case=False, na=False)]
    df_camas = df_camas[(df_camas['INDICE_OCUPACIONAL'] >= 0) & (df_camas['INDICE_OCUPACIONAL'] <= 120)]
    
    # Filtro maestro aplanado usando todas las variaciones del diccionario
    all_sss = [item for sublist in mapa_sss.values() for item in sublist]
    df_camas = df_camas[df_camas['GLOSA_SSS'].isin(all_sss)]
except:
    df_camas = pd.DataFrame()

coords = {
    'Osorno': {'lat': -40.5739, 'lon': -73.1336},
    'Chiguayante': {'lat': -36.9209, 'lon': -73.0253},
    'Concepcion': {'lat': -36.8201, 'lon': -73.0444},
    'Temuco': {'lat': -38.7359, 'lon': -72.5904},
    'Valdivia': {'lat': -39.8196, 'lon': -73.2452}
}

meses_nombres = {1:'Enero', 2:'Febrero', 3:'Marzo', 4:'Abril', 5:'Mayo', 6:'Junio', 7:'Julio', 8:'Agosto', 9:'Septiembre', 10:'Octubre', 11:'Noviembre', 12:'Diciembre'}
meses_iniciales = {1:'E', 2:'F', 3:'M', 4:'A', 5:'M', 6:'J', 7:'J', 8:'A', 9:'S', 10:'O', 11:'N', 12:'D'}

Cargando y procesando datos...


# Layout del Dashboard  

In [23]:
app = dash.Dash(__name__)

estilo_tarjeta = {
    'backgroundColor': 'white', 'padding': '20px', 'borderRadius': '10px',
    'boxShadow': '0 4px 6px rgba(0,0,0,0.1)', 'marginBottom': '20px'
}
estilo_desc = {'fontSize': '13px', 'color': '#7f8c8d', 'textAlign': 'center', 'marginTop': '-10px', 'marginBottom': '15px'}

app.layout = html.Div(style={'fontFamily': 'Arial, sans-serif', 'backgroundColor': '#f4f6f9', 'padding': '30px'}, children=[
    
    html.Div(style=estilo_tarjeta, children=[
        html.H1("Dashboard: Saturación Hospitalaria, Virus y Calidad del Aire (PM 2.5)", style={'color': '#2c3e50', 'textAlign': 'center'}),
        html.Div(style={'backgroundColor': '#e8f4f8', 'padding': '15px', 'borderRadius': '8px', 'borderLeft': '5px solid #3498db'}, children=[
            html.H3("Objetivo de la Visualización", style={'marginTop': '0', 'color': '#2980b9'}),
            html.P([html.B("Problema: "), "Explorar la relación entre los niveles de contaminación atmosférica (PM 2.5), el aumento de atenciones de urgencia respiratorias y la ocupación de camas críticas en el sur de Chile."]),
            html.P([html.B("Usuario: "), "Analistas de Salud Pública, Gestores Hospitalarios y Seremis de Salud."])
        ])
    ]),

    html.Div(style=estilo_tarjeta, children=[
        html.H4("Filtros Globales", style={'color': '#2c3e50'}),
        html.Div(style={'display': 'flex', 'gap': '20px'}, children=[
            html.Div(style={'flex': '1'}, children=[
                html.Label("Seleccione Año:"),
                dcc.Dropdown(id='filtro-ano', options=[{'label': str(ano), 'value': ano} for ano in range(2020, 2026)], value=2020, clearable=False)
            ]),
            html.Div(style={'flex': '1'}, children=[
                html.Label("Seleccione Comuna:"),
                dcc.Dropdown(id='filtro-comuna', options=[{'label': c, 'value': c} for c in ciudades_pm], value='Osorno', clearable=False)
            ])
        ])
    ]),

    # Fila 1
    html.Div(style={'display': 'flex', 'gap': '20px'}, children=[
        html.Div(style={**estilo_tarjeta, 'flex': '1'}, children=[
            html.H4(id='t-mapa', style={'textAlign': 'center'}),
            html.P("Mapa con marcadores escalado por el nivel de contaminación ambiental histórico.", style=estilo_desc),
            dcc.Graph(id='grafico-mapa')
        ]),
        html.Div(style={**estilo_tarjeta, 'flex': '1'}, children=[
            html.H4(id='t-comp', style={'textAlign': 'center'}),
            html.P("Composición de diagnósticos para priorización de recursos e insumos médicos.", style=estilo_desc),
            dcc.Graph(id='grafico-composicion')
        ])
    ]),

    # Fila 2
    html.Div(style=estilo_tarjeta, children=[
        html.H4(id='t-serie', style={'textAlign': 'center'}),
        html.P("Evolución temporal semanal. Permite observar si los peaks de PM 2.5 anteceden la saturación de urgencias.", style=estilo_desc),
        dcc.Graph(id='grafico-serie-tiempo')
    ]),

    # Fila 3
    html.Div(style={'display': 'flex', 'gap': '20px'}, children=[
        html.Div(style={**estilo_tarjeta, 'flex': '1'}, children=[
            html.H4(id='t-barras', style={'textAlign': 'center'}),
            html.P("Promedio de urgencias semanales agrupadas según la clasificación normativa de calidad del aire.", style=estilo_desc),
            dcc.Graph(id='grafico-barras-impacto')
        ]),
        html.Div(style={**estilo_tarjeta, 'flex': '1'}, children=[
            html.H4(id='t-heat', style={'textAlign': 'center'}),
            html.P("Matriz de correlación (Pearson) entre variables medioambientales y carga asistencial.", style=estilo_desc),
            dcc.Graph(id='grafico-heatmap')
        ])
    ]),

    # Fila 4
    html.Div(style={'display': 'flex', 'gap': '20px'}, children=[
        html.Div(style={**estilo_tarjeta, 'flex': '1'}, children=[
            html.H4(id='t-dist', style={'textAlign': 'center'}),
            html.P("Histograma: Promedio de saturación de camas críticas según los distintos rangos de contaminación del aire.", style=estilo_desc),
            dcc.Graph(id='grafico-distribucion') 
        ]),
        html.Div(style={**estilo_tarjeta, 'flex': '1'}, children=[
            html.H4(id='t-camas', style={'textAlign': 'center'}),
            html.P("Camas físicas reales ocupadas (Rojo) vs. camas libres (Gris). Evidencia la complejización invernal.", style=estilo_desc),
            dcc.Graph(id='grafico-camas-serie')
        ])
    ]),

    # SECCIÓN DE FUENTES
    html.Hr(style={'marginTop': '40px', 'marginBottom': '20px', 'borderColor': '#bdc3c7'}),
    html.Div(style={'textAlign': 'left', 'fontSize': '12px', 'color': '#7f8c8d'}, children=[
        html.B("Fuentes y Referencias Bibliográficas:"),
        html.Ul(style={'marginTop': '10px'}, children=[
            html.Li(["Impacto cuarentenas COVID-19 en transmisión viral: ", html.A("Resumen.cl / Datos ISP", href="https://resumen.cl/articulos/muertes-por-enfermedades-respiratorias-han-disminuido-un-34-durante-la-pandemia", target="_blank", style={'color': '#3498db'})]),
            html.Li(["Complejización y reconversión de camas críticas: ", html.A("Ministerio de Salud - Gobierno de Chile", href="https://www.gob.cl/noticias/salud-instruye-aumentar-numero-de-camas-criticas-un-30-por-sobre-del-maximo-habilitado-en-la-primera-ola/", target="_blank", style={'color': '#3498db'})])
        ])
    ])
])

# Gráficos y ejecucución de la app 

In [24]:
@app.callback(
    [
     Output('grafico-mapa', 'figure'), Output('grafico-composicion', 'figure'), Output('grafico-serie-tiempo', 'figure'),
     Output('grafico-barras-impacto', 'figure'), Output('grafico-heatmap', 'figure'), Output('grafico-distribucion', 'figure'),
     Output('grafico-camas-serie', 'figure'), Output('t-mapa', 'children'), Output('t-comp', 'children'),
     Output('t-serie', 'children'), Output('t-barras', 'children'), Output('t-heat', 'children'),
     Output('t-dist', 'children'), Output('t-camas', 'children')
    ],
    [Input('filtro-ano', 'value'), Input('filtro-comuna', 'value')]
)
def actualizar_dashboard(ano_sel, comuna_sel):
    
    df_u_filt = df_urgencias[(df_urgencias['Año'] == ano_sel) & (df_urgencias['Comuna'] == comuna_sel)] if not df_urgencias.empty else pd.DataFrame()
    df_p_filt = df_pm25[(df_pm25['fecha'].dt.year == ano_sel) & (df_pm25['Comuna'] == comuna_sel)] if not df_pm25.empty else pd.DataFrame()
    
    sss_seleccionado = mapa_sss.get(comuna_sel, [])
    df_c_filt = df_camas[(df_camas['PERIODO'] == ano_sel) & (df_camas['GLOSA_SSS'].isin(sss_seleccionado))] if not df_camas.empty else pd.DataFrame()

    t_map = f"1. Mapa PM 2.5 - ({ano_sel})"
    t_comp = f"2. Composición Causas - {comuna_sel} ({ano_sel})"
    t_serie = f"3. Evolución: Smog vs Urgencias - {comuna_sel} ({ano_sel})"
    t_barras = f"4. Pacientes por Estado del Aire - {comuna_sel} ({ano_sel})"
    t_heat = f"5. Heatmap de Correlación - {comuna_sel} ({ano_sel})"
    t_dist = f"6. Histograma: Saturación vs PM 2.5 - {comuna_sel} ({ano_sel})"
    t_camas = f"7. Camas Físicas (Ocupadas vs Libres) - Nivel Regional ({ano_sel})"

    # 1. Mapa
    map_data = [{'Comuna': c, 'Lat': latlon['lat'], 'Lon': latlon['lon'], 'PM25 Promedio': df_pm25[(df_pm25['fecha'].dt.year == ano_sel) & (df_pm25['Comuna'] == c)]['PM25'].mean() if not pd.isna(df_pm25[(df_pm25['fecha'].dt.year == ano_sel) & (df_pm25['Comuna'] == c)]['PM25'].mean()) else 10} for c, latlon in coords.items()]
    fig_map = px.scatter_mapbox(pd.DataFrame(map_data), lat="Lat", lon="Lon", size="PM25 Promedio", color="PM25 Promedio", hover_name="Comuna", color_continuous_scale=px.colors.sequential.YlOrRd, size_max=25, zoom=5)
    fig_map.update_layout(mapbox_style="carto-positron", margin={"r":0,"t":0,"l":0,"b":0})

    # 2. Composición (¡AQUÍ SE AGREGÓ LA LEYENDA INFERIOR!)
    if not df_u_filt.empty and 'GlosaCausa' in df_u_filt.columns:
        df_causas = df_u_filt.groupby('GlosaCausa')['Total'].sum().reset_index().sort_values('Total', ascending=False).head(5)
        fig_comp = px.pie(df_causas, names='GlosaCausa', values='Total', hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
    else:
        fig_comp = go.Figure().add_annotation(text="Sin datos", showarrow=False)
        
    # Activamos la leyenda, la ponemos horizontal, y le damos margen inferior para que no aplaste el gráfico
    fig_comp.update_layout(
        margin=dict(t=20, b=70, l=20, r=20), 
        showlegend=True, 
        legend=dict(
            orientation="h", 
            yanchor="top", 
            y=-0.1, 
            xanchor="center", 
            x=0.5, 
            font=dict(size=10)
        )
    )

    # 3. Evolución 
    fig_ts = go.Figure()
    if not df_p_filt.empty and not df_u_filt.empty:
        df_p_week = df_p_filt.sort_values('fecha').set_index('fecha').resample('W')['PM25'].mean().reset_index()
        df_u_week = df_u_filt.sort_values('fecha').set_index('fecha').resample('W')['Total'].sum().reset_index()
        fig_ts.add_trace(go.Scatter(x=df_p_week['fecha'], y=df_p_week['PM25'], name='PM 2.5', line=dict(color='firebrick', width=2), yaxis='y1', connectgaps=True))
        fig_ts.add_trace(go.Scatter(x=df_u_week['fecha'], y=df_u_week['Total'].fillna(0), name='Urgencias', line=dict(color='royalblue', width=2), yaxis='y2', connectgaps=True))
        if ano_sel in [2020, 2021]:
            fig_ts.add_annotation(x=pd.to_datetime(f'{ano_sel}-06-15'), y=df_p_week['PM25'].max() if df_p_week['PM25'].max() > 0 else 50, yref="y1", text="<b>Observación:</b><br>Cuarentenas COVID-19 redujeron la transmisión viral.", showarrow=True, ax=0, ay=-40, bgcolor="rgba(255, 255, 255, 0.9)", bordercolor="#34495e", borderwidth=1, borderpad=4, font=dict(size=11, color="#2c3e50"))
    fig_ts.update_layout(xaxis=dict(title='Meses', dtick="M1", tickformat="%b"), yaxis=dict(title=dict(text='Nivel PM 2.5', font=dict(color='firebrick')), tickfont=dict(color='firebrick')), yaxis2=dict(title=dict(text='Urgencias', font=dict(color='royalblue')), tickfont=dict(color='royalblue'), anchor='x', overlaying='y', side='right'), margin=dict(t=30, b=30, l=30, r=30), legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5))

    # 4. Barras de Impacto
    if not df_p_filt.empty and not df_u_filt.empty:
        df_corr = pd.merge(df_p_filt.groupby(df_p_filt['fecha'].dt.isocalendar().week)['PM25'].mean().reset_index(), df_u_filt.groupby('semana')['Total'].sum().reset_index(), left_on='week', right_on='semana')
        df_corr['Calidad Aire'] = pd.cut(df_corr['PM25'], bins=[-1, 50, 80, 110, 170, 999], labels=['Bueno', 'Regular', 'Alerta', 'Preemergencia', 'Emergencia'])
        fig_barras = px.bar(df_corr.groupby('Calidad Aire', observed=False)['Total'].mean().reset_index(), x='Calidad Aire', y='Total', text_auto='.0f', color='Calidad Aire', color_discrete_map={'Bueno':'#2ecc71', 'Regular':'#f1c40f', 'Alerta':'#e67e22', 'Preemergencia':'#e74c3c', 'Emergencia':'#8e44ad'})
        fig_barras.update_traces(textposition="outside", cliponaxis=False)
    else:
        fig_barras = go.Figure().add_annotation(text="Sin datos", showarrow=False)
    fig_barras.update_layout(margin=dict(t=20, b=20, l=20, r=20), showlegend=False, xaxis_title="Calidad Aire", yaxis_title="Pacientes Semanal")

    # 5. HEATMAP 
    if not df_p_filt.empty and not df_u_filt.empty and not df_c_filt.empty:
        df_matrix = pd.merge(df_p_filt.groupby(df_p_filt['fecha'].dt.month)['PM25'].mean().reset_index(), df_u_filt.groupby('Mes')['Total'].sum().reset_index(), left_on='fecha', right_on='Mes')
        df_matrix = pd.merge(df_matrix, df_c_filt.groupby('MES')['INDICE_OCUPACIONAL'].mean().reset_index(), left_on='Mes', right_on='MES').rename(columns={'PM25': 'PM 2.5', 'Total': 'Urgencias', 'INDICE_OCUPACIONAL': 'Saturación (%)'})
        fig_heat = px.imshow(df_matrix[['PM 2.5', 'Urgencias', 'Saturación (%)']].corr(), text_auto=".2f", aspect="auto", color_continuous_scale='RdBu_r')
        fig_heat.update_layout(margin=dict(t=20, b=20, l=20, r=20))
    else:
        fig_heat = go.Figure().add_annotation(text="Sin datos", showarrow=False)

    # 6. HISTOGRAMA 
    if not df_c_filt.empty and not df_pm25.empty:
        df_p_macro = df_p_filt.groupby(df_p_filt['fecha'].dt.month)['PM25'].mean().reset_index()
        df_p_macro.rename(columns={'fecha': 'MES'}, inplace=True)
        df_hist = pd.merge(df_c_filt, df_p_macro, on='MES', how='inner')
        df_hist = df_hist.dropna(subset=['PM25', 'INDICE_OCUPACIONAL'])
        
        fig_box = px.histogram(df_hist, x="PM25", y="INDICE_OCUPACIONAL", histfunc="avg", nbins=6,
                               labels={"PM25": "Rango Nivel PM 2.5 Mensual", "INDICE_OCUPACIONAL": "Ocupación Promedio (%)"},
                               color_discrete_sequence=['#c0392b'])
        
        fig_box.update_layout(bargap=0.1, margin=dict(t=20, b=20, l=20, r=20), 
                              yaxis_title="Ocupación Promedio (%)",
                              yaxis=dict(range=[0, 110]))
    else:
        fig_box = go.Figure().add_annotation(text="Sin datos", showarrow=False)

    # 7. CAMAS APILADAS 
    if not df_c_filt.empty:
        df_c_grp = df_c_filt.groupby('MES')[['DIAS_CAMAS_OCUPADAS', 'DIAS_CAMAS_DISPONIBLES']].sum().reset_index()
        df_c_grp['Camas Libres'] = (df_c_grp['DIAS_CAMAS_DISPONIBLES'] - df_c_grp['DIAS_CAMAS_OCUPADAS']).clip(lower=0) 
        df_c_grp['Mes_Init'] = df_c_grp['MES'].map(meses_iniciales)

        fig_camas = go.Figure()
        fig_camas.add_trace(go.Bar(x=df_c_grp['Mes_Init'], y=df_c_grp['DIAS_CAMAS_OCUPADAS'], name='Camas Ocupadas', marker_color='#e74c3c'))
        fig_camas.add_trace(go.Bar(x=df_c_grp['Mes_Init'], y=df_c_grp['Camas Libres'], name='Camas Libres', marker_color='#bdc3c7'))
        
        fig_camas.update_layout(barmode='stack', margin=dict(t=30, b=20, l=20, r=20), xaxis_title="Meses", yaxis_title="Total Días Cama", legend=dict(orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5))
    else:
        fig_camas = go.Figure().add_annotation(text="Sin datos", showarrow=False)

    return fig_map, fig_comp, fig_ts, fig_barras, fig_heat, fig_box, fig_camas, t_map, t_comp, t_serie, t_barras, t_heat, t_dist, t_camas

if __name__ == '__main__':
    app.run(debug=True, port=8050, jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/


/var/folders/pl/b0xfp74x5v52p3m732fd7jw00000gn/T/ipykernel_1314/2608705899.py:29: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_map = px.scatter_mapbox(pd.DataFrame(map_data), lat="Lat", lon="Lon", size="PM25 Promedio", color="PM25 Promedio", hover_name="Comuna", color_continuous_scale=px.colors.sequential.YlOrRd, size_max=25, zoom=5)
/var/folders/pl/b0xfp74x5v52p3m732fd7jw00000gn/T/ipykernel_1314/2608705899.py:29: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_map = px.scatter_mapbox(pd.DataFrame(map_data), lat="Lat", lon="Lon", size="PM25 Promedio", color="PM25 Promedio", hover_name="Comuna", color_continuous_scale=px.colors.sequential.YlOrRd, size_max=25, zoom=5)
/var/folders/pl/b0xfp74x5v52p3m732fd7jw00000gn/T/ipykernel_1314/2608705899.py:29: DeprecationWarning: *scatter_mapbox* is depr